# Volume indicator – SPARQL notebook

Dit notebook laat zien hoe de volume indicator kan worden berekend met behulp van SPARQL queries op een RDF graph die is opgebouwd volgens de TI-O ontologie. 
We definiëren eerst de patientcohort, en vervolgens gebruiken we deze om het behandelvolume, en het aantal behandelde patienten per locatie te berekenen.

In [2]:
%pip install rdflib --quiet

Note: you may need to restart the kernel to use updated packages.


# Laadt de ontologie en voorbeelddata in een RDF graph

In [3]:
from rdflib import Graph, Literal, URIRef
import pandas as pd

g = Graph()
g.parse("../transparantie_indicatoren.owl.ttl", format="turtle")
g.parse("THP_example_data.ttl", format="turtle")
print(f"Graph loaded with {len(g)} triples")

Graph loaded with 734 triples


# Definieer het patientcohort query

In [4]:

patientCohortQueryDeel = """
        # --- Tijdsbinding: begindatum van de behandeling valt binnen verslagperiode ---
        ?behandeling time:hasTime/time:hasBeginning/time:inXSDDate ?datumBehandeling .
        FILTER ( ?datumBehandeling >= ?verslagPeriodeStart && ?datumBehandeling < ?verslagPeriodeEind )

        # --- Patient met aandoening die wordt behandeld ---
        ?behandeling
            a ?behandelingType ;
            obo:RO_0000057 ?patient ;   # 'has participant' (heeft als deelnemer)
            obo:RO_0000057 ?disorder .

        ?patient
            a obo:NCBITAXON_9606 ;
            obo:BFO_00000137 ?disorder . # 'has continuant part at some time' (de disorder is van de deelnemende persoon)

        # --- Exclusie: behandeling heeft geen deelnemende disorder
        #     die aangeduid wordt door een exclusie klasse ---
        FILTER NOT EXISTS { ?disorder a ?exclusie . }
    """

bindings = {
    "verslagPeriodeStart": Literal("2023-01-01", datatype="http://www.w3.org/2001/XMLSchema#date"),
    "verslagPeriodeEind": Literal("2024-01-01", datatype="http://www.w3.org/2001/XMLSchema#date"),
    "behandelingType": URIRef("https://w3id.org/zinl/ti-o#THP"),
    "exclusie": URIRef("https://w3id.org/zinl/ti-o#HipFracture")
}


# A. Gebruikt de patientcohortdefinitie voor het berekenen van het behandelvolume per locatie

In [10]:
treatmentVolumeQuery = f"""
PREFIX obo:     <http://purl.obolibrary.org/obo/>
PREFIX time:    <http://www.w3.org/2006/time#>
PREFIX ti-o:    <https://w3id.org/zinl/ti-o#>
PREFIX xsd:     <http://www.w3.org/2001/XMLSchema#>

SELECT ?locatie (COUNT(DISTINCT ?behandeling) AS ?aantal)
WHERE {{

    # Cohortdeel wordt hier ingevoegd:
    {patientCohortQueryDeel}
    
    # --- Zorglocatie ---
    ?behandeling obo:BFO_0000066 ?locatie . # 'occurs in' (vindt plaats in)
}}
GROUP BY ?locatie
ORDER BY ?locatie
"""

results = g.query(treatmentVolumeQuery, initBindings=bindings)

pd.DataFrame(results, columns=results.vars)

,locatie,aantal
0,https://w3id.org/zinl/data/example-volume#Ziek...,2


# B. Gebruik de patientcohortdefinitie voor het berekenen van het aantal behandelde patienten per locatie

In [6]:
patientVolumeQuery = f"""
PREFIX obo:     <http://purl.obolibrary.org/obo/>
PREFIX time:    <http://www.w3.org/2006/time#>
PREFIX ti-o:    <https://w3id.org/zinl/ti-o#>
PREFIX xsd:     <http://www.w3.org/2001/XMLSchema#>

SELECT ?locatie (COUNT(DISTINCT ?patient) AS ?aantal)
WHERE {{

    # Cohortdeel wordt hier ingevoegd:
    {patientCohortQueryDeel}
    
    # --- Zorglocatie ---
    ?behandeling obo:BFO_0000066 ?locatie . # 'occurs in' (vindt plaats in)
}}
GROUP BY ?locatie
ORDER BY ?locatie
"""

results = g.query(patientVolumeQuery, initBindings=bindings)

pd.DataFrame(results, columns=results.vars)

,locatie,aantal
0,https://w3id.org/zinl/data/example-volume#Ziek...,2
